<a href="https://colab.research.google.com/github/enslern/The-AI-engineer-course-2026-Complete-AI-bootcamp/blob/main/pinecone_semantic_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**data loading and preprocessing**

In [ ]:
import pandas as pd

In [ ]:
files = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/resources/course_descriptions.csv", encoding='latin1')

In [ ]:
def create_course_description(row):
  return f'''The course name is {row["course_name"]} , the slug is {row["course_slug"]} ,
  the technology is {row["course_technology"]} and the topic is {row["course_topic"]}'''

In [ ]:
files["course_description_new"]=files.apply(create_course_description,axis=1)

In [ ]:
files["course_description_new"]

,course_description_new
0,"The course name is Introduction to Tableau , t..."
1,The course name is The Complete Data Visualiza...
2,The course name is Introduction to R Programmi...
3,The course name is Data Preprocessing with Num...
4,The course name is Introduction to Data and Da...
...,...
101,"The course name is Intro to NLP for AI , the s..."
102,The course name is Data Analysis with ChatGPT ...
103,"The course name is ChatGPT for Data Science , ..."
104,"The course name is Intro to LLMs , the slug is..."


In [ ]:
files

,course_name,course_slug,course_technology,course_description,course_topic,course_description_short,course_description_new
0,Introduction to Tableau,tableau,tableau,Tableau is now one of the most popular busines...,data visualization,Teaching you how to tell compelling stories wi...,"The course name is Introduction to Tableau , t..."
1,The Complete Data Visualization Course with Py...,data-visualization,python,The Data Visualization course is designed for ...,data visualization,Teaching you how to master the art of creating...,The course name is The Complete Data Visualiza...
2,Introduction to R Programming,introduction-to-r-programming,r,R is one of the best programming languages spe...,programming,"Providing you with the skills to manipulate, a...",The course name is Introduction to R Programmi...
3,Data Preprocessing with NumPy,data-preprocessing-numpy,python,This course is designed to show you how to wor...,data processing,This course will guide you through one of Pyth...,The course name is Data Preprocessing with Num...
4,Introduction to Data and Data Science,intro-to-data-and-data-science,theory,Working with data is an essential part of main...,machine learning,Introducing you to the field of data science a...,The course name is Introduction to Data and Da...
...,...,...,...,...,...,...,...
101,Intro to NLP for AI,intro-to-nlp-for-ai,python,Natural language processing is an exciting and...,programming,Unlock the power of natural language processin...,"The course name is Intro to NLP for AI , the s..."
102,Data Analysis with ChatGPT,data-analysis-with-chatgpt,chatgpt,Leverage ChatGPT's Advanced Data Analysis Code...,programming,Master ChatGPT for data analysis. Boost your p...,The course name is Data Analysis with ChatGPT ...
103,ChatGPT for Data Science,chatgpt-for-data-science,chatgpt,"In this course, you will learn how to utilize ...",machine learning,Learn how to increase your productivity using ...,"The course name is ChatGPT for Data Science , ..."
104,Intro to LLMs,intro-to-llms,python,"In recent years, large language models (LLMs) ...",machine learning,This LLM course will guide you step-by-step th...,"The course name is Intro to LLMs , the slug is..."


##**connecting to the pinecone server**

In [ ]:
import os
import sys

In [ ]:
sys.path.append(os.path.abspath("/content/drive/MyDrive/Colab Notebooks/resources"))

In [ ]:
import config

In [ ]:
os.environ["PINECONE_API_KEY"]=config.PINECONE_API_KEY
os.environ["PINECONE_ENV"]=config.PINECONE_ENV

In [ ]:
!pip install pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 9.3 MB/s eta 0:00:00


In [ ]:
import pinecone
from pinecone import Pinecone,ServerlessSpec

In [ ]:
pc=Pinecone()

In [ ]:
index_name="my-index"
dimension=384
metric="cosine"

In [ ]:
if index_name in [index.name for index in pc.list_indexes()]:
  pc.delete_index(name=index_name)
  print(f"{index_name} was successfully deleted")
else:
  print(f"{index_name} does not exist")


my-index was successfully deleted


In [ ]:
pc.create_index(
    name=index_name,
    dimension=dimension,
    metric=metric,
    spec=ServerlessSpec(
        region="us-east-1",
        cloud="aws"
    )
)

Name:,my-index
Status:,Ready
Ready:,Yes
Deployment:,Managed (aws/us-east-1)
Host:,https://my-index-toy62j7.svc.aped-4627-b74a.pinecone.io
Deletion Protection:,disabled
Schema fields:,2
Read capacity:,"ReadCapacityOnDemandResponse(status=ReadCapacityStatus(state='Ready', current_shards=None, current_replicas=None, error_message=None))"


##**embedding the files and upserting to pinecone server**

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model=SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def create_embeddings(row):
  combined_text="".join([str(row[field]) for field in ["course_description","course_description_new","course_description_short"]])
  embedding=model.encode(combined_text,show_progress_bar=False)
  return embedding

In [ ]:
files["embeddings"]=files.apply(create_embeddings,axis=1)

In [ ]:
files["embeddings"]

,embeddings
0,"[0.038174838, -0.030057695, -0.08009775, -0.00..."
1,"[0.047157302, -0.028943455, -0.012238161, -0.0..."
2,"[-0.046136796, 0.004143676, -0.08703885, 0.032..."
3,"[-0.053299617, -0.014942402, -0.048904654, -0...."
4,"[-0.027056186, 0.02446643, -0.0069476715, 0.02..."
...,...
101,"[-0.029041486, -0.04720483, 0.029386338, 0.007..."
102,"[-0.048341423, 0.024535274, -0.016557705, -0.0..."
103,"[-0.10137507, -0.014043052, 0.027810914, 0.051..."
104,"[-0.040015984, -0.10109687, 0.044334345, -0.02..."


In [ ]:
vectors_to_upsert=[(str(row["course_name"]),row["embeddings"].tolist())     for _ ,row in files.iterrows()]


In [ ]:
index=pc.Index(index_name)
index.upsert(vectors=vectors_to_upsert)

UpsertResponse(upserted_count=106)

##**Semantic search**

In [ ]:
query="clustering"
query_embedding=model.encode(query,show_progress_bar=False).tolist()

In [ ]:
query_result=index.query(
    vector=[query_embedding],
    top_k=12,
    include_values=True
)

In [ ]:
query_result

QueryResponse(matches=[ScoredVector(id='Machine Learning in Excel', score=0.363712341, values=[-0.0131248524, -0.0299022961, -0.0260536838, ...381 more]), ScoredVector(id='Machine Learning with K-Nearest Neighbors', score=0.326225311, values=[-0.108806923, -0.0832003728, 0.0136599271, ...381 more]), ScoredVector(id='Machine Learning in Python', score=0.283647567, values=[-0.0496008359, -0.0432145484, -0.00637800712, ...381 more]), ScoredVector(id='Customer Churn Analysis with SQL and Tableau', score=0.283303261, values=[-0.0260646623, -0.0535058938, -0.0688754, ...381 more]), ScoredVector(id='Linear Algebra and Feature Selection', score=0.260547668, values=[-0.037694, -0.0168079697, 0.0155882044, ...381 more]), ScoredVector(id='Growth Analysis with SQL, Python, and Tableau  ', score=0.253543884, values=[0.020653218, -0.032449767, -0.0757706463, ...381 more]), ScoredVector(id='Customer Engagement Analysis with SQL and Tableau', score=0.242667213, values=[0.037405733, -0.0126662, -0.0531

In [28]:
for match in query_result["matches"]:
  print(f"Matched item id:{match["id"]},score:{match["score"]}")

Matched item id:Machine Learning in Excel,score:0.363712341
Matched item id:Machine Learning with K-Nearest Neighbors,score:0.326225311
Matched item id:Machine Learning in Python,score:0.283647567
Matched item id:Customer Churn Analysis with SQL and Tableau,score:0.283303261
Matched item id:Linear Algebra and Feature Selection,score:0.260547668
Matched item id:Growth Analysis with SQL, Python, and Tableau  ,score:0.253543884
Matched item id:Customer Engagement Analysis with SQL and Tableau,score:0.242667213
Matched item id:Fashion Analytics with Tableau,score:0.236726776
Matched item id:Machine Learning with Support Vector Machines,score:0.233779937
Matched item id:Data Analysis with Excel Pivot Tables,score:0.220921531
Matched item id:Machine Learning with Naive Bayes,score:0.211374313
Matched item id:Data Preprocessing with NumPy,score:0.199216858
